# RAG pipeline walkthrough

This notebook breaks Retrieval-Augmented Generation (RAG) into independent steps so each stage is easy to inspect and run:
1. Setup and environment
2. Document loading
3. Chunking
4. Embeddings and vector store
5. Query embedding and retrieval
6. Reranking

In [ ]:
# ---------------- 0. Environment and .env ----------------
# If packages are missing, uncomment and run:
# !pip install -q chromadb langchain-community langchain-openai langchain-chroma sentence-transformers python-dotenv

import os
from dotenv import load_dotenv

# Load API keys, e.g. OPENAI_API_KEY
load_dotenv(dotenv_path="../.env")

print("OpenAI API Key:", "set" if os.environ.get("OPENAI_API_KEY") else "not set (required for embeddings below)")
print("If you see errors, ensure ../.env contains OPENAI_API_KEY.")

### 1. Document loading
For this demo we write a small text file in the current directory and load it with LangChain `TextLoader`. In production you might use `PyPDFLoader`, `UnstructuredWordDocumentLoader`, etc.

In [ ]:
# ---------------- 1. Read file ----------------
from langchain_community.document_loaders import TextLoader

# Build a small sample file for RAG
sample_text = """
Artificial Intelligence (AI) is a branch of computer science that studies the nature of intelligence and builds machines that can react in ways similar to human intelligence.
Large Language Models (LLMs) are a major advance in deep learning: trained on massive data, they gain broad understanding and strong text generation.
Retrieval-Augmented Generation (RAG) combines retrieval with generation by pulling relevant content from an external knowledge base, improving accuracy and freshness of answers.
Vector databases such as Chroma or FAISS store and efficiently retrieve high-dimensional vectors at scale.
"""
file_path = "sample_knowledge.txt"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(sample_text.strip())

# Step 1: load with TextLoader
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

print(f"Loaded documents: {len(documents)}")
print("---- First document preview ----")
print(documents[0].page_content[:150] + "...")

### 2. Chunking
Large files must be split into chunks sized for embedding limits and for useful retrieval granularity.

In [ ]:
# ---------------- 2. Document chunking ----------------
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Small chunk size (70 chars) to show splitting clearly on this short sample
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=70,      # max characters per chunk
    chunk_overlap=15    # overlap to preserve context across boundaries
)

chunked_docs = text_splitter.split_documents(documents)

print(f"Split {len(documents)} document(s) into {len(chunked_docs)} chunks.\n")
for i, chunk in enumerate(chunked_docs):
    print(f"[Chunk {i+1}] (length: {len(chunk.page_content)})")
    print(chunk.page_content)
    print("-" * 40)

### 3 & 4. Embeddings and vector store
Embed the chunks and store vectors in a lightweight local Chroma collection.

In [ ]:
# ---------------- 3 & 4. Embeddings + Chroma ----------------
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Step 3: build embeddings model
# Swap in another embedding class here if you use Zhipu, BGE, etc.
embeddings_model = OpenAIEmbeddings()

# Step 4: index chunks in Chroma (in-memory; use persist_directory to persist)
vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings_model,
    collection_name="test_rag_collection"
)

print(f"Indexed {len(chunked_docs)} chunks in Chroma (in-memory).")
print("Restarting the kernel clears the in-memory store.")

### 5 & 6. Query embedding and retrieval
Embed the user query and similarity-search the vector store.

In [ ]:
# ---------------- 5 & 6. Query embedding + retrieval ----------------
user_query = "What is RAG?"

# Step 5 (demo): show query embedding shape
query_embedding = embeddings_model.embed_query(user_query)
print(f"Query '{user_query}' embeds to a vector of length {len(query_embedding)}.\n")

# Step 6: retrieve top-k chunks
# similarity_search embeds the query then finds nearest neighbors
top_k = 3
retrieved_docs = vector_store.similarity_search(user_query, k=top_k)

print(f"--- Vector retrieval (top {top_k}) ---")
for i, doc in enumerate(retrieved_docs):
    print(f"\nChunk {i+1}:\n{doc.page_content}")

### 7. Reranking
Vector similarity can return chunks that look similar but miss deeper intent.

Here we use a `CrossEncoder` (or another reranker) to score `(query, document)` pairs and reorder results.

In [ ]:
# ---------------- 7. Reranking ----------------
try:
    from sentence_transformers import CrossEncoder
    import numpy as np

    print("Loading reranker (first run may download weights)...")
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

    pairs = [[user_query, doc.page_content] for doc in retrieved_docs]
    scores = reranker.predict(pairs)
    print("Reranker scores:", scores)

    sorted_indices = np.argsort(scores)[::-1]

    print("\n================ Reranked results ================")
    for new_rank, idx in enumerate(sorted_indices):
        original_rank = idx + 1
        score = scores[idx]
        print(f"[Rank {new_rank+1}] (original rank: {original_rank}, score: {score:.4f}):")
        print(retrieved_docs[idx].page_content)
        print("-" * 50)

except ImportError:
    print("[Missing dependency]")
    print("`sentence_transformers` not found. Install with:")
    print("!pip install sentence-transformers")
    print("\n--- Mock rerank demo ---")
    print("[Mock] Reverse original order to show ranking can change with finer relevance:")
    for i, doc in enumerate(reversed(retrieved_docs)):
        print(f"[Reranked {i+1}]:\n{doc.page_content}\n")